### Load dataset, helper methods


In [5]:
import pandas as pd
from articles_dataset import ArticlesDataset
articles_dataset_path = \
    "../resources/bcc-news-entertainment.csv"
dataset = ArticlesDataset(articles_dataset_path)
dataset.load_articles(50)
dataset.preprocess()

sents_count = 0
for article in dataset.articles:
    sents_count += len(list(article.document.sents))
print("Total sentences count acrros loaded documents: ", sents_count)

Total sentences count acrros loaded documents:  743


In [6]:

import re
import hashlib

def readable_id(sentence: str, length: int = 4) -> str:
    h = hashlib.md5(sentence.encode()).hexdigest()[:length]
    slug = re.sub(r'\W+', '_', sentence.lower()).strip('_')[:30]

    return f"{slug}_{h}"

### NLTK based approach

In [3]:
from nltk.sentiment import SentimentIntensityAnalyzer
vader_analyzer = SentimentIntensityAnalyzer()

In [4]:

rows = []

for article in dataset.articles:
    for sentence in article.document.sents:
        # print(type(sentence.text))
        scores = vader_analyzer.polarity_scores(sentence.text)
        rows.append({
            "sentence_id": readable_id(sentence.text),
            "sentence": sentence.text,
            "neg": scores["neg"],
            "neu": scores["neu"],
            "pos": scores["pos"],
            "compound": scores["compound"],
        })

vader_output_df = pd.DataFrame(rows)

In [5]:
vader_output_df

,sentence_id,sentence,neg,neu,pos,compound
0,french_musician_jean_michel_ja_702a,French musician Jean-Michel Jarre is to perfo...,0.000,1.000,0.000,0.0000
1,denmark_is_holding_a_three_day_22dc,Denmark is holding a three-day celebration of ...,0.000,1.000,0.000,0.0000
2,other_stars_are_expected_to_jo_f264,Other stars are expected to join the line-up i...,0.000,0.891,0.109,0.2960
3,christian_andersen_s_fairy_tal_0356,"""Christian Andersen's fairy tales are timeless...",0.000,1.000,0.000,0.0000
4,for_all_of_us_at_any_age_there_1692,"""For all of us, at any age there is always - b...",0.000,0.841,0.159,0.5574
...,...,...,...,...,...,...
738,if_i_m_half_the_person_you_are_1bf8,If I'm half the person you are and half the ta...,0.000,0.787,0.213,0.6908
739,both_foxx_and_swank_are_now_co_34ae,Both Foxx and Swank are now considered to be a...,0.000,0.845,0.155,0.5106
740,however_swank_has_to_overcome__6e3c,"However, Swank has to overcome a strong challe...",0.127,0.657,0.216,0.2263
741,meanwhile_ballots_for_oscars_t_728e,"Meanwhile, ballots for Oscars - the Hollywood'...",0.000,0.815,0.185,0.6124


### Hugging face approach (cardiffnlp/twitter-roberta-base-sentiment)

In [6]:
from transformers import AutoModelForSequenceClassification, AutoTokenizer, pipeline

TASK = "sentiment"
MODEL = f"cardiffnlp/twitter-roberta-base-{TASK}"


tokenizer = AutoTokenizer.from_pretrained(MODEL)
model = AutoModelForSequenceClassification.from_pretrained(MODEL)


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: cardiffnlp/twitter-roberta-base-sentiment
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [7]:
import urllib
import csv

# download label mapping
labels=[]
mapping_link = f"https://raw.githubusercontent.com/cardiffnlp/tweeteval/main/datasets/{TASK}/mapping.txt"
with urllib.request.urlopen(mapping_link) as f:
    html = f.read().decode('utf-8').split("\n")
    csvreader = csv.reader(html, delimiter='\t')
labels = [row[1] for row in csvreader if len(row) > 1]

In [8]:
labels

['negative', 'neutral', 'positive']

In [9]:
from scipy.special import softmax

rows = []
for article in dataset.articles:
    for sentence in article.document.sents:
        encoded_input = tokenizer(sentence.text, return_tensors='pt')
        output = model(**encoded_input)
        scores = output[0][0].detach().numpy()
        scores = softmax(scores)
        rows.append({
            "sentence_id": readable_id(sentence.text),
            "sentence": sentence.text,
            "neg": scores[0],
            "neu": scores[1],
            "pos": scores[2],
            "compound": None,
        })

model_output_df = pd.DataFrame(rows)

In [10]:
model_output_df

,sentence_id,sentence,neg,neu,pos,compound
0,french_musician_jean_michel_ja_702a,French musician Jean-Michel Jarre is to perfo...,0.006829,0.866350,0.126821,None
1,denmark_is_holding_a_three_day_22dc,Denmark is holding a three-day celebration of ...,0.001329,0.301878,0.696792,None
2,other_stars_are_expected_to_jo_f264,Other stars are expected to join the line-up i...,0.002328,0.674368,0.323304,None
3,christian_andersen_s_fairy_tal_0356,"""Christian Andersen's fairy tales are timeless...",0.011130,0.484049,0.504821,None
4,for_all_of_us_at_any_age_there_1692,"""For all of us, at any age there is always - b...",0.022180,0.428832,0.548987,None
...,...,...,...,...,...,...
738,if_i_m_half_the_person_you_are_1bf8,If I'm half the person you are and half the ta...,0.012914,0.148072,0.839014,None
739,both_foxx_and_swank_are_now_co_34ae,Both Foxx and Swank are now considered to be a...,0.001499,0.145952,0.852549,None
740,however_swank_has_to_overcome__6e3c,"However, Swank has to overcome a strong challe...",0.102957,0.785562,0.111481,None
741,meanwhile_ballots_for_oscars_t_728e,"Meanwhile, ballots for Oscars - the Hollywood'...",0.024945,0.885263,0.089792,None


In [11]:
df = model_output_df.merge(
    vader_output_df,
    on="sentence_id",
    suffixes=("_model", "_vader")
)
df = df.drop(columns=["sentence_vader"])
df = df.rename(columns={"sentence_model": "sentence"})

In [12]:
df.head()

,sentence_id,sentence,neg_model,neu_model,pos_model,compound_model,neg_vader,neu_vader,pos_vader,compound_vader
0,french_musician_jean_michel_ja_702a,French musician Jean-Michel Jarre is to perfo...,0.006829,0.866350,0.126821,None,0.0,1.000,0.000,0.0000
1,denmark_is_holding_a_three_day_22dc,Denmark is holding a three-day celebration of ...,0.001329,0.301878,0.696792,None,0.0,1.000,0.000,0.0000
2,other_stars_are_expected_to_jo_f264,Other stars are expected to join the line-up i...,0.002328,0.674368,0.323304,None,0.0,0.891,0.109,0.2960
3,christian_andersen_s_fairy_tal_0356,"""Christian Andersen's fairy tales are timeless...",0.011130,0.484049,0.504821,None,0.0,1.000,0.000,0.0000
4,for_all_of_us_at_any_age_there_1692,"""For all of us, at any age there is always - b...",0.022180,0.428832,0.548987,None,0.0,0.841,0.159,0.5574


In [13]:
df

,sentence_id,sentence,neg_model,neu_model,pos_model,compound_model,neg_vader,neu_vader,pos_vader,compound_vader
0,french_musician_jean_michel_ja_702a,French musician Jean-Michel Jarre is to perfo...,0.006829,0.866350,0.126821,None,0.000,1.000,0.000,0.0000
1,denmark_is_holding_a_three_day_22dc,Denmark is holding a three-day celebration of ...,0.001329,0.301878,0.696792,None,0.000,1.000,0.000,0.0000
2,other_stars_are_expected_to_jo_f264,Other stars are expected to join the line-up i...,0.002328,0.674368,0.323304,None,0.000,0.891,0.109,0.2960
3,christian_andersen_s_fairy_tal_0356,"""Christian Andersen's fairy tales are timeless...",0.011130,0.484049,0.504821,None,0.000,1.000,0.000,0.0000
4,for_all_of_us_at_any_age_there_1692,"""For all of us, at any age there is always - b...",0.022180,0.428832,0.548987,None,0.000,0.841,0.159,0.5574
...,...,...,...,...,...,...,...,...,...,...
744,if_i_m_half_the_person_you_are_1bf8,If I'm half the person you are and half the ta...,0.012914,0.148072,0.839014,None,0.000,0.787,0.213,0.6908
745,both_foxx_and_swank_are_now_co_34ae,Both Foxx and Swank are now considered to be a...,0.001499,0.145952,0.852549,None,0.000,0.845,0.155,0.5106
746,however_swank_has_to_overcome__6e3c,"However, Swank has to overcome a strong challe...",0.102957,0.785562,0.111481,None,0.127,0.657,0.216,0.2263
747,meanwhile_ballots_for_oscars_t_728e,"Meanwhile, ballots for Oscars - the Hollywood'...",0.024945,0.885263,0.089792,None,0.000,0.815,0.185,0.6124


import pandas as pd
pd.read_csv("/results/sentiment_analysis_nltk_vs_model.csv")
### Hugging face approach (base)